# KL divergence vs. $L^r$ distances between probability densities

MATH 166 — Statistics, Fall 2026 (Prof. Andreas Mang)

We compare two ways of measuring how different two densities $p$ and $q$ on $\mathbb{R}$ are:

* the **$L^r$ distance** (for densities; $\ell^r$ for probability mass functions)
$$
\|p-q\|_r = \Bigl(\int_{\mathbb{R}} |p(x)-q(x)|^{r}\,dx\Bigr)^{1/r},
\qquad
\|p-q\|_\infty = \sup_x |p(x)-q(x)|,
$$
  where $\|p-q\|_1 = 2\,\mathrm{TV}(p,q)$ is twice the total variation distance;
* the **Kullback–Leibler (KL) divergence**
$$
\mathrm{KL}(p\,\|\,q) = \int_{\mathbb{R}} p(x)\,\log\frac{p(x)}{q(x)}\,dx
= \mathbb{E}_{p}\Bigl[\log\frac{p(X)}{q(X)}\Bigr] \;\ge\; 0 .
$$

Four observations, one per section:

1. **Shifted Gaussians.** $L^r$ distances *saturate* once the densities stop overlapping; KL keeps growing.
2. **Scale.** KL and $L^1$ do not change if we rescale $x$ (change units); $L^2$ and $L^\infty$ do.
3. **Asymmetry.** $\mathrm{KL}(p\,\|\,q)\ne\mathrm{KL}(q\,\|\,p)$ in general; a Gaussian against a two-component mixture shows why it matters (*mode covering* vs. *mode seeking*).
4. **Support.** KL can be infinite while every $L^r$ distance is finite. Finally, maximum likelihood is minimizing a KL divergence.

Requires only `numpy` and `matplotlib`. All integrals are computed numerically on a fine grid and checked against closed forms where they exist.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

# ---- plot style: thin lines, recessive grid, colour-blind-safe series colours
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({
    "figure.dpi": 110, "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "axes.titlesize": 11,
    "axes.titleweight": "bold", "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "xtick.color": INK2, "ytick.color": INK2, "text.color": INK,
    "lines.linewidth": 2.0, "legend.frameon": False, "font.size": 10,
})

# ---- the grid on which we integrate
x = np.linspace(-40, 40, 16001)

def integrate(f):
    """Trapezoidal rule on the grid x."""
    return np.trapezoid(f, x)

def log_normal(x, mu=0.0, sigma=1.0):
    """log of the N(mu, sigma^2) density."""
    return -0.5 * np.log(2 * np.pi * sigma**2) - (x - mu)**2 / (2 * sigma**2)

def log_mixture(x, weights, mus, sigmas):
    """log of a Gaussian mixture density, computed stably (log-sum-exp)."""
    comps = np.array([np.log(w) + log_normal(x, m, s)
                      for w, m, s in zip(weights, mus, sigmas)])
    cmax = comps.max(axis=0)
    return cmax + np.log(np.exp(comps - cmax).sum(axis=0))

def kl(logp, logq):
    """KL(p || q) = int p log(p/q) dx, from log densities (0 log 0 = 0)."""
    p = np.exp(logp)
    with np.errstate(invalid="ignore", divide="ignore"):
        integrand = np.where(p > 0, p * (logp - logq), 0.0)
    return integrate(integrand)

def lr_dist(logp, logq, r):
    """L^r distance ||p - q||_r (r = np.inf gives the sup distance)."""
    d = np.abs(np.exp(logp) - np.exp(logq))
    return d.max() if r == np.inf else integrate(d**r) ** (1 / r)

def Phi(z):
    """Standard normal CDF."""
    return 0.5 * (1 + math.erf(z / math.sqrt(2)))

## 1. Two Gaussians shifted against each other

Let $p = \mathcal{N}(0,1)$ and $q = \mathcal{N}(\mu,1)$. Here everything has a closed form:
$$
\mathrm{KL}(p\,\|\,q) = \mathrm{KL}(q\,\|\,p) = \frac{\mu^{2}}{2},
\qquad
\|p-q\|_1 = 2\bigl(2\Phi(\mu/2)-1\bigr)\;\to\; 2,
\qquad
\|p-q\|_2 = \sqrt{\frac{1-e^{-\mu^{2}/4}}{\sqrt{\pi}}}\;\to\;\pi^{-1/4}.
$$
(With equal variances the KL divergence happens to be symmetric; Section 3 breaks that.)

In [ ]:
mus_show = [0.5, 2.0, 5.0]
fig, axes = plt.subplots(1, 3, figsize=(10, 2.6), sharey=True)
for ax, mu in zip(axes, mus_show):
    p, q = np.exp(log_normal(x, 0, 1)), np.exp(log_normal(x, mu, 1))
    ax.fill_between(x, np.minimum(p, q), color=GRID, label="overlap")
    ax.plot(x, p, color=BLUE, label=r"$p=\mathcal{N}(0,1)$")
    ax.plot(x, q, color=ORANGE, label=r"$q=\mathcal{N}(\mu,1)$")
    ax.set_xlim(-4, 9); ax.set_title(rf"$\mu = {mu}$"); ax.set_xlabel("$x$")
axes[0].set_ylabel("density")
axes[0].legend(loc="upper right")
fig.tight_layout(); plt.show()

In [ ]:
shifts = np.linspace(0, 8, 161)
logp = log_normal(x, 0, 1)
res = {"KL": [], "L1": [], "L2": [], "Linf": []}
for mu in shifts:
    logq = log_normal(x, mu, 1)
    res["KL"].append(kl(logp, logq))
    res["L1"].append(lr_dist(logp, logq, 1))
    res["L2"].append(lr_dist(logp, logq, 2))
    res["Linf"].append(lr_dist(logp, logq, np.inf))
res = {k: np.array(v) for k, v in res.items()}

# check the numerics against the closed forms
kl_exact = shifts**2 / 2
l1_exact = np.array([2 * (2 * Phi(mu / 2) - 1) for mu in shifts])
l2_exact = np.sqrt((1 - np.exp(-shifts**2 / 4)) / np.sqrt(np.pi))
print("max error  KL: %.1e   L1: %.1e   L2: %.1e" % (
    abs(res["KL"] - kl_exact).max(), abs(res["L1"] - l1_exact).max(),
    abs(res["L2"] - l2_exact).max()))

Each $L^r$ distance has a largest possible value, reached when the two densities no longer overlap: then $\|p-q\|_r^r = \|p\|_r^r + \|q\|_r^r$. To compare the three on one axis, we divide each by that limit, so every curve ends at $1$.

In [ ]:
lim = {"L1": 2.0, "L2": np.pi ** -0.25, "Linf": res["Linf"][-1]}   # values at full separation
fig, (axL, axK) = plt.subplots(1, 2, figsize=(10, 3.4))

for key, col, lab in [("L1", BLUE, "$L^1$"), ("L2", ORANGE, "$L^2$"), ("Linf", AQUA, r"$L^\infty$")]:
    axL.plot(shifts, res[key] / lim[key], color=col, label=lab)
axL.set_title(r"$L^r$ distance, as a fraction of its maximum")
axL.set_xlabel(r"shift $\mu$"); axL.set_ylim(0, 1.05); axL.legend(loc="lower right")

axK.plot(shifts, res["KL"], color=BLUE, label=r"$\mathrm{KL}(p\,\|\,q)$")
axK.set_title("KL divergence")
axK.set_xlabel(r"shift $\mu$"); axK.set_ylim(bottom=0)
axK.annotate(r"$\mu^2/2$, unbounded", xy=(6.5, 6.5**2 / 2), xytext=(2.2, 22),
             color=INK2, arrowprops=dict(arrowstyle="-", color=INK2))
fig.tight_layout(); plt.show()

**What to take away.**

* Every $L^r$ distance compares the *heights* of the two curves point by point. Once the bumps stop overlapping (around $\mu\approx 4$), the distance is at its maximum: $L^r$ cannot tell "far apart" from "very far apart". For $L^1$ this has a clean meaning: $\tfrac12\|p-q\|_1 = \mathrm{TV}(p,q) = \sup_A|P(A)-Q(A)|$, and once there is an event $A$ with $P(A)\approx 1$ and $Q(A)\approx 0$, nothing larger is possible.
* KL compares the *log-ratio* $\log\bigl(p(x)/q(x)\bigr)$, averaged over $p$. Where $q$ is tiny but $p$ is not, that ratio is huge, so KL keeps growing (here like $\mu^{2}/2$).

## 2. Changing units

Take the same picture at three different scales: $p = \mathcal{N}(0,\sigma^2)$, $q = \mathcal{N}(\sigma,\sigma^2)$, so the shift is always one standard deviation. This is the *same* pair of distributions measured in different units (say metres vs. centimetres).

In [ ]:
print(f"{'sigma':>6} {'KL(p||q)':>10} {'L1':>8} {'L2':>8} {'Linf':>8}")
for sigma in [0.5, 1.0, 2.0, 4.0]:
    lp, lq = log_normal(x, 0, sigma), log_normal(x, sigma, sigma)
    print(f"{sigma:6.1f} {kl(lp, lq):10.4f} {lr_dist(lp, lq, 1):8.4f} "
          f"{lr_dist(lp, lq, 2):8.4f} {lr_dist(lp, lq, np.inf):8.4f}")

KL and $L^1$ do not change: both are invariant under a change of variables $y = g(x)$, so they measure a property of the *distributions*, not of how we write them down. $L^2$ scales like $\sigma^{-1/2}$ and $L^\infty$ like $\sigma^{-1}$, because a density has units of $1/x$.

## 3. Asymmetry of the KL divergence

### 3a. Same mean, different spread

Let $p = \mathcal{N}(0,1)$ and $q = \mathcal{N}(0,\sigma^2)$. Then
$$
\mathrm{KL}(p\,\|\,q) = \log\sigma + \frac{1}{2\sigma^{2}} - \frac12,
\qquad
\mathrm{KL}(q\,\|\,p) = -\log\sigma + \frac{\sigma^{2}}{2} - \frac12 .
$$

In [ ]:
sigmas = np.geomspace(0.2, 5, 201)
logp = log_normal(x, 0, 1)
kl_pq = np.array([kl(logp, log_normal(x, 0, s)) for s in sigmas])
kl_qp = np.array([kl(log_normal(x, 0, s), logp) for s in sigmas])
print("max error vs closed form: %.1e, %.1e" % (
    abs(kl_pq - (np.log(sigmas) + 1 / (2 * sigmas**2) - 0.5)).max(),
    abs(kl_qp - (-np.log(sigmas) + sigmas**2 / 2 - 0.5)).max()))

fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(sigmas, kl_pq, color=BLUE, label=r"$\mathrm{KL}(p\,\|\,q)$")
ax.plot(sigmas, kl_qp, color=ORANGE, label=r"$\mathrm{KL}(q\,\|\,p)$")
ax.set_xscale("log"); ax.set_ylim(0, 6)
ax.set_xticks([0.2, 0.5, 1, 2, 5]); ax.set_xticklabels(["0.2", "0.5", "1", "2", "5"])
ax.set_xlabel(r"standard deviation $\sigma$ of $q$ (log scale)")
ax.set_title(r"$p=\mathcal{N}(0,1)$ vs. $q=\mathcal{N}(0,\sigma^2)$")
ax.legend(loc="upper center")
fig.tight_layout(); plt.show()

$\mathrm{KL}(p\,\|\,q)$ punishes a $q$ that is too *narrow* ($\sigma<1$): then $q$ is tiny in the tails of $p$, exactly where $\log(p/q)$ is large and $p$ still has mass. $\mathrm{KL}(q\,\|\,p)$ punishes a $q$ that is too *wide*. Every $L^r$ distance is symmetric by definition, so no such distinction exists there.

### 3b. A Gaussian against a two-component mixture

Now let $p$ be bimodal,
$$
p = \tfrac12\,\mathcal{N}(-3,1) + \tfrac12\,\mathcal{N}(3,1),
$$
and slide a single Gaussian $q = \mathcal{N}(m,1)$ across it.

In [ ]:
d = 3.0
logp = log_mixture(x, [0.5, 0.5], [-d, d], [1, 1])
ms = np.linspace(-8, 8, 321)
fwd = np.array([kl(logp, log_normal(x, m, 1)) for m in ms])        # KL(p || q)
rev = np.array([kl(log_normal(x, m, 1), logp) for m in ms])        # KL(q || p)
l1 = np.array([lr_dist(logp, log_normal(x, m, 1), 1) for m in ms])

fig, (ax0, ax1, ax2) = plt.subplots(1, 3, figsize=(12, 3.4))
ax0.plot(x, np.exp(logp), color=BLUE, label="$p$ (mixture)")
ax0.plot(x, np.exp(log_normal(x, 0, 1)), color=ORANGE, label="$q$ at $m=0$")
ax0.plot(x, np.exp(log_normal(x, d, 1)), color=AQUA, label="$q$ at $m=3$")
ax0.set_xlim(-8, 8); ax0.set_xlabel("$x$"); ax0.set_title("the densities")
ax0.legend(loc="upper left", fontsize=8)

ax1.plot(ms, fwd, color=BLUE, label=r"$\mathrm{KL}(p\,\|\,q)$")
ax1.plot(ms, rev, color=ORANGE, label=r"$\mathrm{KL}(q\,\|\,p)$")
ax1.set_ylim(0, 12); ax1.set_xlabel("center $m$ of $q$")
ax1.set_title("KL, both directions"); ax1.legend(loc="upper center")

ax2.plot(ms, l1, color=BLUE)
ax2.set_ylim(0, 2.05); ax2.set_xlabel("center $m$ of $q$"); ax2.set_title(r"$L^1$ distance")
fig.tight_layout(); plt.show()

print("KL(p||q) is smallest at m = %.2f" % ms[fwd.argmin()])
print("KL(q||p) is smallest at m = %+.2f (and its mirror image)" % ms[rev.argmin()])

* $\mathrm{KL}(p\,\|\,q)$ averages over $p$, so it is large wherever $p$ has mass and $q$ does not. The best $q$ sits **between** the two modes, where $p$ itself has almost no mass, because that is the only place from which $q$ reaches both bumps. This is called **mode covering**.
* $\mathrm{KL}(q\,\|\,p)$ averages over $q$, so it only cares that $q$ puts no mass where $p$ is small. The best $q$ sits **on one of the modes** and ignores the other. This is called **mode seeking**.
* $L^1$ has no preferred direction; like $\mathrm{KL}(q\,\|\,p)$ it prefers $q$ on a mode here.

### 3c. Best Gaussian fit in each direction

Let both the mean $m$ and the spread $s$ of $q = \mathcal{N}(m, s^2)$ vary, and find the best fit to the mixture under each KL direction.

In [ ]:
m_grid = np.linspace(-5, 5, 201)
s_grid = np.linspace(0.5, 4.5, 161)
F = np.empty((s_grid.size, m_grid.size)); R = np.empty_like(F)
p = np.exp(logp)
for i, s in enumerate(s_grid):
    logq = log_normal(x[None, :], m_grid[:, None], s)             # one row per m
    q = np.exp(logq)
    F[i] = np.trapezoid(p * (logp - logq), x, axis=1)              # KL(p || q)
    R[i] = np.trapezoid(q * (logq - logp), x, axis=1)              # KL(q || p)

i, j = np.unravel_index(F.argmin(), F.shape); m_f, s_f = m_grid[j], s_grid[i]
i, j = np.unravel_index(R.argmin(), R.shape); m_r, s_r = m_grid[j], s_grid[i]
print(f"argmin KL(p||q): m = {m_f:+.2f}, s = {s_f:.2f}   "
      f"(mean and sd of p: 0, sqrt(1+d^2) = {np.sqrt(1 + d**2):.2f})")
print(f"argmin KL(q||p): m = {m_r:+.2f}, s = {s_r:.2f}")

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(x, p, color=BLUE, label="$p$ (mixture)")
ax.plot(x, np.exp(log_normal(x, m_f, s_f)), color=ORANGE,
        label=rf"best for $\mathrm{{KL}}(p\,\|\,q)$: $\mathcal{{N}}({m_f:.1f}, {s_f:.2f}^2)$")
ax.plot(x, np.exp(log_normal(x, m_r, s_r)), color=AQUA,
        label=rf"best for $\mathrm{{KL}}(q\,\|\,p)$: $\mathcal{{N}}({m_r:.1f}, {s_r:.2f}^2)$")
ax.set_xlim(-9, 9); ax.set_ylim(0, 0.42); ax.set_xlabel("$x$")
ax.set_title("mode covering vs. mode seeking"); ax.legend(loc="upper left", fontsize=8)
fig.tight_layout(); plt.show()

Minimizing $\mathrm{KL}(p\,\|\,q)$ over Gaussians matches the **mean and variance** of $p$ (a general fact for exponential families): a broad Gaussian over both modes. Minimizing $\mathrm{KL}(q\,\|\,p)$ picks **one mode** and fits it tightly. Which direction is "right" depends on what the fit is for; this is why the order of the arguments must always be stated.

## 4. When KL is infinite

Let $p = \text{Uniform}(-1,1)$ and $q = \mathcal{N}(0,1)$. Then
$$
\mathrm{KL}(p\,\|\,q) = -\log 2 + \tfrac12\log(2\pi) + \tfrac16 \approx 0.392,
\qquad
\mathrm{KL}(q\,\|\,p) = \infty,
$$
because $q$ puts mass outside $[-1,1]$, where $p = 0$ and $\log(q/p) = \infty$. In general, $\mathrm{KL}(p\,\|\,q) < \infty$ requires $q > 0$ wherever $p > 0$. The $L^r$ distances never blow up like this.

In [ ]:
with np.errstate(divide="ignore"):
    log_unif = np.where(np.abs(x) <= 1, np.log(0.5), -np.inf)
log_gauss = log_normal(x, 0, 1)
print("KL(p||q) = %.3f   (exact %.3f)" % (kl(log_unif, log_gauss),
      -np.log(2) + 0.5 * np.log(2 * np.pi) + 1 / 6))
print("KL(q||p) =", kl(log_gauss, log_unif))
print("L1 = %.3f,  L2 = %.3f" % (lr_dist(log_unif, log_gauss, 1), lr_dist(log_unif, log_gauss, 2)))

(The numerical $\mathrm{KL}(p\,\|\,q)$ is off in the third digit because the uniform density jumps at $\pm1$, which the trapezoidal rule smears over one grid cell.)

## 5. Connection to maximum likelihood

For data $X_1,\dots,X_n \overset{\text{iid}}{\sim} p$ and a model $q_\theta$, the average log-likelihood is
$$
\frac1n\sum_{i=1}^{n}\log q_\theta(X_i) \;\approx\; \mathbb{E}_p\bigl[\log q_\theta(X)\bigr]
= -\mathrm{KL}(p\,\|\,q_\theta) + \mathbb{E}_p\bigl[\log p(X)\bigr],
$$
and the last term does not depend on $\theta$. So **maximizing the likelihood is (approximately) minimizing $\mathrm{KL}(p\,\|\,q_\theta)$**, the mode-covering direction. Fitting a Gaussian by maximum likelihood to data from the mixture should therefore reproduce the broad fit of Section 3c.

In [ ]:
rng = np.random.default_rng(166)
n = 5000
labels = rng.random(n) < 0.5
sample = np.where(labels, rng.normal(-d, 1, n), rng.normal(d, 1, n))
mu_hat = sample.mean()
sd_hat = np.sqrt(np.mean((sample - mu_hat) ** 2))     # Gaussian MLE of sigma
print(f"Gaussian MLE from {n} samples: mu = {mu_hat:+.2f}, sigma = {sd_hat:.2f}")
print(f"argmin of KL(p||q) (Section 3c): m = {m_f:+.2f}, s = {s_f:.2f}")

## Things to try

* In Section 1, give $q$ a different variance from $p$. Is KL still symmetric?
* In Section 3b, move the modes closer together ($d = 1$). When does $\mathrm{KL}(q\,\|\,p)$ stop having two minima?
* Make the mixture weights unequal (say $0.8$ and $0.2$). Which mode does the reverse-KL fit choose?
* Replace $q$ in Section 4 by a Student-$t$ or Laplace density. Which KL directions are finite now?